# Clean scholar's career data
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing and loading all dependencies and libraries needed

In [9]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

## Checking names numbers

In [4]:
import pandas as pd

# Step 1: Load CSV files
# Replace 'path_to_csv1' and 'path_to_csv2' with the actual paths in your Google Drive.
csv1_path = '/content/drive/My Drive/clean/cleaner/master_dataset.csv'
csv2_path = '/content/drive/My Drive/clean/cleaner/temporarycareer.csv'


In [8]:
# Load the CSV files into DataFrames
df1 = pd.read_csv(csv1_path)
df2 = pd.read_csv(csv2_path)

# Step 2: Check the 'Name' column in both CSVs
# Ensure the 'Name' column exists in both CSVs
if 'Name' in df1.columns and 'Name' in df2.columns:
    # Find the matches
    matches = set(df1['Name']).intersection(set(df2['Name']))
    match_count = len(matches)

    # Print the number of matching names
    print(f"Number of matching names: {match_count}")

    # Find non-matching names in df2
    non_matching_names = set(df2['Name']) - set(df1['Name'])
    non_matching_count = len(non_matching_names)

    # Print the non-matching names and their count
    print(f"Number of names in df2 but not in df1: {non_matching_count}")
    print("Names in df2 but not in df1:")
    print(non_matching_names)
else:
    print("The 'Name' column is missing in one or both CSV files.")


Number of matching names: 867
Number of names in df2 but not in df1: 0
Names in df2 but not in df1:
set()


In [6]:
# Step 3: Check the 'Name' column in both CSVs
if 'Name' in df1.columns and 'Name' in df2.columns:
    # Determine names in df2 but not in df1
    df1_names = set(df1['Name'])
    df2_names = set(df2['Name'])
    names_in_df2_not_in_df1 = df2_names - df1_names

    # Count and list the names
    count = len(names_in_df2_not_in_df1)
    print(f"Number of names in df2 but not in df1: {count}")
    print("Names in df2 but not in df1:")
    print(names_in_df2_not_in_df1)
else:
    print("The 'Name' column is missing in one or both CSV files.")

Number of names in df2 but not in df1: 0
Names in df2 but not in df1:
set()


In [7]:
# Step 3: Count the number of rows in each DataFrame
df1_rows = len(df1)
df2_rows = len(df2)

# Print the number of rows in each DataFrame
print(f"Number of rows in df1: {df1_rows}")
print(f"Number of rows in df2: {df2_rows}")

Number of rows in df1: 2636
Number of rows in df2: 886


## Merging unsorted career data

In [12]:
# Step 1: Ensure the 'Name' column exists in both DataFrames
if 'Name' in df1.columns and 'Name' in df2.columns:
    # Step 2: Find matching names
    matching_names = set(df1['Name']).intersection(set(df2['Name']))

    # Step 3: Filter df2 rows with matching names
    df2_matching_rows = df2[df2['Name'].isin(matching_names)]

    # Step 4: Copy specified columns in order
    columns_to_copy = [
        "firstName", "lastName", "fullName", "description","headline","location",
        "connectionsCount","jobTitle", "company", "jobLocation", "jobDateRange", "jobDuration",
        "jobStartedSince", "jobIsCurrent", "companyUrl", "companyDomainName",
        "company2", "jobTitle2", "jobLocation2", "jobDateRange2", "jobDuration2",
        "jobStartedSince2", "jobIsCurrent2", "school", "schoolDegree",
        "schoolDescription", "schoolDateRange", "schoolUrl", "school2",
        "schoolDegree2", "schoolDateRange2", "schoolUrl2", "phoneNumberFromDropContact", "websiteFromDropContact",
        "website", "companyWebsite", "professionalEmail", "subscribers",
        "jobDescription", "mailFromDropcontact"
    ]

    # Ensure all specified columns exist in df2
    missing_columns = [col for col in columns_to_copy if col not in df2.columns]
    if missing_columns:
        print(f"The following columns are missing in df2: {missing_columns}")
    else:
        # Copy the columns and append them to df2
        new_columns = df2_matching_rows[columns_to_copy].reset_index(drop=True)

        # Add the new columns to df2
        df1_extended = pd.concat([df1.reset_index(drop=True), new_columns], axis=1)

        # Sort columns: Keep original `df2` columns first, then the reordered new columns
        reordered_columns = list(df1.columns) + columns_to_copy
        df1_extended = df1_extended[reordered_columns]

        # Print a confirmation message
        print("Columns appended and sorted successfully. Updated DataFrame:")
        print(df1_extended.head())  # Display the first few rows of the updated DataFrame

        # Optionally save the updated DataFrame
        output_path = '/content/drive/My Drive/clean/cleaner/career_extended_unsorted.csv'
        df1_extended.to_csv(output_path, index=False)
        print(f"Updated DataFrame saved to: {output_path}")
else:
    print("The 'Name' column is missing in one or both CSV files.")


Columns appended and sorted successfully. Updated DataFrame:
               Name revised_gender UniversityType                 Institution  \
0       A Damodaran           Male        Private        Ahmedabad University   
1  A Lavanya Kumari         Female          State  Adikavi Nannaya University   
2     A V Arunkumar           Male        Private               RV University   
3        A. Anbuvel           Male          State        Annamalai University   
4       A. Jayakodi           Male          State        Annamalai University   

                                        archive_link  \
0  http://web.archive.org/web/20240928200657/http...   
1  http://web.archive.org/web/20240801140914/http...   
2  http://web.archive.org/web/20240922111038/http...   
3  http://web.archive.org/web/20240805150230/http...   
4  http://web.archive.org/web/20240805150230/http...   

                              college_url  \
0                  https://ahduni.edu.in/   
1                    http

In [11]:
# List of specific columns
specific_columns = [
    "description", "headline", "location", "firstName", "lastName", "fullName",
    "connectionsCount", "company", "companyUrl", "companyDomainName", "jobTitle",
    "jobLocation", "jobDateRange", "jobStartedSince", "jobIsCurrent", "jobDuration",
    "company2", "companyUrl2", "jobTitle2", "jobLocation2", "jobDateRange2",
    "jobStartedSince2", "jobIsCurrent2", "jobDuration2", "school", "schoolUrl",
    "schoolDegree", "schoolDescription", "schoolDateRange", "school2",
    "schoolUrl2", "schoolDegree2", "schoolDateRange2", "phoneNumberFromDropContact",
    "websiteFromDropContact", "website", "companyWebsite", "professionalEmail",
    "subscribers", "jobDescription", "mailFromDropcontact"
]

# Step 1: Count NaN and empty values in specific columns in df2
print("Counts of NaN and empty values in df2:")
for column in specific_columns:
    num_nans = df2[column].isna().sum()
    num_empty = (df2[column] == '').sum()
    print(f"{column}: NaNs = {num_nans}, Empty = {num_empty}")

# Step 2: Assuming df2 has already been extended with new columns (merged dataset)
df1_extended = pd.concat([df2, df2[specific_columns]], axis=1)

# Recheck NaN and empty values in specific columns in the merged dataset
print("\nCounts of NaN and empty values in the merged dataset:")
for column in specific_columns:
    num_nans = df1_extended[column].isna().sum()
    num_empty = (df1_extended[column] == '').sum()
    print(f"{column}: NaNs = {num_nans}, Empty = {num_empty}")


Counts of NaN and empty values in df2:
description: NaNs = 612, Empty = 0
headline: NaNs = 133, Empty = 0
location: NaNs = 133, Empty = 0
firstName: NaNs = 133, Empty = 0
lastName: NaNs = 133, Empty = 0
fullName: NaNs = 133, Empty = 0
connectionsCount: NaNs = 136, Empty = 0
company: NaNs = 191, Empty = 0
companyUrl: NaNs = 351, Empty = 0
companyDomainName: NaNs = 306, Empty = 0
jobTitle: NaNs = 189, Empty = 0
jobLocation: NaNs = 458, Empty = 0
jobDateRange: NaNs = 317, Empty = 0
jobStartedSince: NaNs = 318, Empty = 0
jobIsCurrent: NaNs = 318, Empty = 0
jobDuration: NaNs = 318, Empty = 0
company2: NaNs = 391, Empty = 0
companyUrl2: NaNs = 509, Empty = 0
jobTitle2: NaNs = 392, Empty = 0
jobLocation2: NaNs = 554, Empty = 0
jobDateRange2: NaNs = 395, Empty = 0
jobStartedSince2: NaNs = 396, Empty = 0
jobIsCurrent2: NaNs = 396, Empty = 0
jobDuration2: NaNs = 399, Empty = 0
school: NaNs = 315, Empty = 0
schoolUrl: NaNs = 443, Empty = 0
schoolDegree: NaNs = 354, Empty = 0
schoolDescription: Na

In [13]:
# Drop 'firstName' and 'lastName' columns from df2
df1_extended = df1_extended.drop(columns=["firstName", "lastName"])

# Print the updated df2 to confirm the columns are dropped
print("\nUpdated df2 after dropping 'firstName' and 'lastName':")
print(df1_extended.head())



Updated df2 after dropping 'firstName' and 'lastName':
               Name revised_gender UniversityType                 Institution  \
0       A Damodaran           Male        Private        Ahmedabad University   
1  A Lavanya Kumari         Female          State  Adikavi Nannaya University   
2     A V Arunkumar           Male        Private               RV University   
3        A. Anbuvel           Male          State        Annamalai University   
4       A. Jayakodi           Male          State        Annamalai University   

                                        archive_link  \
0  http://web.archive.org/web/20240928200657/http...   
1  http://web.archive.org/web/20240801140914/http...   
2  http://web.archive.org/web/20240922111038/http...   
3  http://web.archive.org/web/20240805150230/http...   
4  http://web.archive.org/web/20240805150230/http...   

                              college_url  \
0                  https://ahduni.edu.in/   
1                    https://a

## Save dataset

In [14]:
# Save the cleaned and updated DataFrame to a new CSV
df1_extended.to_csv("/content/drive/My Drive/clean/cleaner/career_extended_unsorted.csv", index=False)

# Experimental

In [ ]:
import spacy
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Function to process text with NLP and add markers
def add_markers_with_nlp(text):
    doc = nlp(text)
    updated_text = []
    delimiter = "***NEW PAPER***"

    session_re = re.compile(r'(Session \d+[a-z]* [^\n]* \([^\)]*\)\n)')
    session_blocks = session_re.split(text)

    for i, block in enumerate(session_blocks):
        if i % 2 == 0:
            updated_text.append(block)
        else:
            papers = re.split(r'(?:"[^"]+"\s*\n)', block)  # Split block at paper titles
            for j, paper in enumerate(papers):
                if j == 0:
                    updated_text.append(paper)
                else:
                    updated_text.append(f'{delimiter} "{papers[j-1].split(": ")[-1].strip()}" {paper}')

    return ''.join(updated_text)

# Assuming 'text' is already loaded
updated_text = add_markers_with_nlp(text)

# Print the updated text
print(updated_text)


Parallel Sessions 1                                                18 December, Monday 11:50 AM- 01:00  PM    
Session 1(a) Social Identity and Financial Inclusion (Conference Room) 
Rikhia Bhukta (Indian Institute of Technology Kanpur), Debayan Pakrashi (Indian Statistical Institute Kolkata; 
Indian Institute of Technology Kanpur), Sarani Saha (Indian Institute of Technology Kanpur), and Ashish Sedai 
(University of Texas at Arlington): "Does Financial Inclusion Mitigate Social Exclusion? Causal Evidence from 
India" 

Sandhya Garg (Institute of Economic Growth), Samarth Gupta (Ahmedabad University), and Sushanta Mallick 
(Queen Mary University of London): "Does Social Identity Constrain Rural Entrepreneurship? The Role of 
Financial Inclusion" 
Session 1(b) Welfare Programmes I (Seminar Room 1) 
Vidhya Unnikrishnan (University of Manchester): "Impact of Social Protection Policy Targeted on the 
Disabled: Panel Data Evidence from India" 


Michael Palmer (University of Western Austral

In [ ]:
import spacy
import pandas as pd
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Define a function to extract session details using NLP
def extract_session_details(text):
    doc = nlp(text)

    sessions = []

    # Regular expression to identify sessions and topics
    session_re = re.compile(r"Session (\d+[a-z]*)([^\n]*)\s*\(([^)]*)\)")

    # Regular expression to match the authors and affiliations
    author_re = re.compile(r"([A-Za-z\s]+)\s?\(([^)]+)\)")

    # Regular expression to capture paper titles
    title_re = re.compile(r'"\*\*\*NEW PAPER\*\*\* ([^\"]+)"')

    # Iterate over all sessions in the text
    for match in session_re.finditer(text):
        session_number, session_topic, location = match.groups()

        # Locate the paper(s) associated with the session
        session_start = match.end()
        session_end = text.find("Session", session_start)  # Find next session to stop the current one
        session_content = text[session_start:session_end] if session_end != -1 else text[session_start:]

        # Extract the paper titles
        paper_titles = title_re.findall(session_content)

        # Extract authors and their affiliations
        authors_and_affiliations = author_re.findall(session_content)

        # Process each paper
        for paper_title in paper_titles:
            authors = []
            affiliations = []

            for author_name, affiliation in authors_and_affiliations:
                authors.append(author_name.strip())
                affiliations.append(affiliation.strip())

            # Loop over authors to create a row for each
            for author, affiliation in zip(authors, affiliations):
                sessions.append({
                    "Session": session_number,
                    "Topic": session_topic,
                    "Title of Paper": paper_title,
                    "Number of Authors": len(authors),
                    "Author": author,
                    "Institute/Affiliation": affiliation
                })

    # Convert the session details into a DataFrame
    df = pd.DataFrame(sessions)

    return df

# Extract session details from the conference text
conference_df = extract_session_details(updated_text)

# Display the structured DataFrame
print(conference_df)

# Save to CSV
conference_df.to_csv("/content/drive/My Drive/conference_details.csv", index=False)


Empty DataFrame
Columns: []
Index: []
